In [1]:
import pandas as pd

In [2]:
retailer_features = pd.read_parquet(
    "../data/processed/retailer_features.parquet"
)

product_features = pd.read_parquet(
    "../data/processed/product_features.parquet"
)

region_trends = pd.read_parquet(
    "../data/processed/region_trends.parquet"
)

interaction = pd.read_parquet(
    "../data/processed/interaction_matrix.parquet"
)

print("Loaded Successfully")

Loaded Successfully


In [3]:
rules = pd.read_csv(
    "../data/processed/association_rules.csv"
)

rules.shape

(92, 16)

In [4]:
sku_lookup = pd.read_parquet(
    "../data/processed/sku_lookup.parquet"
)

sku_lookup.head()

,skuNumber,itemName,brand,category
0,SKU00060,Rajnigandha 4g,Rajnigandha,Pan Masala
1,SKUNK2,Jabsons Green Peas onion & Garlic,Jabsons,Namkeen
2,SKUNK13,Haldiram - Salted Peanuts (â‚¹5),Haldiram,Namkeen
3,SKU501,Haldiram - Aloo Bhujia (24 pcs),Haldiram,Namkeen
4,SKU478,Haldiram - Bhujia (24 pcs),Haldiram,Namkeen


In [5]:
popularity_dict = dict(
    zip(
        product_features["skuNumber"],
        product_features["popularity_score"]
    )
)

In [6]:
similarity_df = pd.read_parquet(
    "../data/processed/similarity_matrix.parquet"
)

similarity_df.shape

(8640, 8640)

In [7]:
def recommend_products(
    customer_id,
    top_n=20
):
    
    similar_users = (
        similarity_df[customer_id]
        .sort_values(
            ascending=False
        )[1:11]
    )

    customer_products = set(
        interaction.loc[
            customer_id
        ][
            interaction.loc[
                customer_id
            ] > 0
        ].index
    )

    recommendations = {}

    for user in similar_users.index:

        similarity_score = (
            similar_users[user]
        )

        purchased = (
            interaction.loc[user]
        )

        for sku in purchased[
            purchased > 0
        ].index:

            if sku not in customer_products:

                recommendations[
                    sku
                ] = (
                    recommendations.get(
                        sku,
                        0
                    )
                    +
                    similarity_score
                )

    recommendations = sorted(
        recommendations.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations[:top_n]

In [8]:
customer_id = interaction.index[0]

cf_recs = recommend_products(
    customer_id,
    top_n=20
)

cf_scores = {
    sku: score
    for sku, score in cf_recs
}

cf_scores

{'SKU00075': np.float64(4.667438394987487),
 'SKUPM5': np.float64(2.799026187806736),
 'SKU00009': np.float64(2.791180622604111),
 'SKU613': np.float64(2.7823346803381876),
 'SKU00001': np.float64(1.866320374063244),
 'SKU00304': np.float64(1.856081648469702),
 'SKU669': np.float64(1.8538801453640883),
 'SKU612': np.float64(1.847055890120161),
 'SKU777': np.float64(0.9352928626146102),
 'SKUCF8': np.float64(0.9352928626146102),
 'SKU00024': np.float64(0.9352787902180266),
 'SKU469': np.float64(0.9352787902180266),
 'SKU615': np.float64(0.9352787902180266),
 'SKU00033': np.float64(0.9310275114486337),
 'SKU483': np.float64(0.9310275114486337),
 'SKU500': np.float64(0.9310275114486337),
 'SKU00020': np.float64(0.9284545349740994),
 'SKU00052': np.float64(0.9284545349740994),
 'SKUCF26': np.float64(0.9284545349740994),
 'SKUHM1': np.float64(0.9284545349740994)}

In [9]:
max_cf = max(
    cf_scores.values()
)

cf_scores = {
    sku: score / max_cf
    for sku, score in cf_scores.items()
}

cf_scores

{'SKU00075': np.float64(1.0),
 'SKUPM5': np.float64(0.5996921546544033),
 'SKU00009': np.float64(0.598011240084422),
 'SKU613': np.float64(0.5961159944448816),
 'SKU00001': np.float64(0.3998596695068425),
 'SKU00304': np.float64(0.3976660196443103),
 'SKU669': np.float64(0.3971943469794974),
 'SKU612': np.float64(0.3957322483578517),
 'SKU777': np.float64(0.20038676110198936),
 'SKUCF8': np.float64(0.20038676110198936),
 'SKU00024': np.float64(0.20038374608702983),
 'SKU469': np.float64(0.20038374608702983),
 'SKU615': np.float64(0.20038374608702983),
 'SKU00033': np.float64(0.19947290840485313),
 'SKU483': np.float64(0.19947290840485313),
 'SKU500': np.float64(0.19947290840485313),
 'SKU00020': np.float64(0.19892164746538415),
 'SKU00052': np.float64(0.19892164746538415),
 'SKUCF26': np.float64(0.19892164746538415),
 'SKUHM1': np.float64(0.19892164746538415)}

In [10]:
customer_profile = retailer_features[
    retailer_features["customerId"]
    == customer_id
]

customer_profile

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order,favorite_category,favorite_brand,spend_segment
0,USR-100,13,41,15,2026-05-27,2026-01-02,Paan A,HVLF,Crossline Events (Noida),4,Pan Masala,DS Group,Medium


In [11]:
hub = customer_profile[
    "hub_name"
].iloc[0]

shop_type = customer_profile[
    "shop_type"
].iloc[0]

print(hub)
print(shop_type)

Crossline Events (Noida)
Paan A


In [12]:
region_recs = region_trends[
    (
        region_trends["hubName"]
        == hub
    )
    &
    (
        region_trends["shopType"]
        == shop_type
    )
]

In [13]:
region_scores = {}

max_rank = (
    region_recs[
        "region_rank"
    ].max()
)

for _, row in region_recs.iterrows():

    score = (
        max_rank -
        row["region_rank"]
    ) / max_rank

    region_scores[
        row["skuNumber"]
    ] = score

len(region_scores)

149

In [14]:
pop_scores = popularity_dict

In [15]:
all_skus = set()

all_skus.update(
    cf_scores.keys()
)

all_skus.update(
    region_scores.keys()
)

all_skus.update(
    pop_scores.keys()
)

In [16]:
hybrid_scores = {}

for sku in all_skus:

    cf = cf_scores.get(
        sku,
        0
    )

    region = region_scores.get(
        sku,
        0
    )

    popularity = pop_scores.get(
        sku,
        0
    )

    final_score = (
        0.6 * cf
        +
        0.25 * region
        +
        0.15 * popularity
    )

    hybrid_scores[
        sku
    ] = final_score
    customer_products = set(
    interaction.loc[
        customer_id
    ][
        interaction.loc[
            customer_id
        ] > 0
    ].index
)

for sku in list(hybrid_scores.keys()):

    if sku in customer_products:

        hybrid_scores.pop(
            sku,
            None
        )

In [17]:
final_recommendations = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

final_recommendations[:10]

[('SKU00075', np.float64(0.8065205938697317)),
 ('SKUPM5', np.float64(0.631061700838619)),
 ('SKU613', np.float64(0.5697265890040937)),
 ('SKU00009', np.float64(0.4423382335142547)),
 ('SKU612', np.float64(0.426491073152642)),
 ('SKU00304', np.float64(0.4131470255796896)),
 ('SKU669', np.float64(0.41113569343674056)),
 ('SKU00033', np.float64(0.3175836492574713)),
 ('SKU00001', np.float64(0.2668446810144503)),
 ('SKU500', np.float64(0.2654428446597701))]

In [18]:
sku_dict = dict(
    zip(
        sku_lookup["skuNumber"],
        sku_lookup["itemName"]
    )
)

for sku, score in final_recommendations[:10]:

    print(
        sku_dict.get(
            sku,
            sku
        ),
        round(score, 3)
    )

Tulsi Royal Khajoor 0.807
Rajnigandha 17g 2 Pcs 0.631
Center Fresh 0.57
Chingles Maxi TF Jar + 1 Rs 5 Lollipop 0.442
Center Fruit 0.426
RG Pearl Elaichi Hanger (6 pcs) 0.413
Alpenliebe Creamfills Butter Toffee 0.411
Rajnigandha 100g  0.318
Chingles Filz Jar + 1 Rs.5 Lollipop 0.267
Haldiram - Salted Peanuts (24 pcs) 0.265


In [19]:
final_recommendations[:10]

[('SKU00075', np.float64(0.8065205938697317)),
 ('SKUPM5', np.float64(0.631061700838619)),
 ('SKU613', np.float64(0.5697265890040937)),
 ('SKU00009', np.float64(0.4423382335142547)),
 ('SKU612', np.float64(0.426491073152642)),
 ('SKU00304', np.float64(0.4131470255796896)),
 ('SKU669', np.float64(0.41113569343674056)),
 ('SKU00033', np.float64(0.3175836492574713)),
 ('SKU00001', np.float64(0.2668446810144503)),
 ('SKU500', np.float64(0.2654428446597701))]

In [20]:
import pandas as pd

similarity_df = pd.read_parquet(
    "../data/processed/similarity_matrix.parquet"
)

similarity_df.shape

(8640, 8640)

In [21]:
import joblib

joblib.dump(
    similarity_df,
    "../models/similarity_matrix.pkl"
)

print("Saved")

Saved


In [22]:
def hybrid_recommend(
    customer_id,
    top_n=10
):
    
    # all hybrid logic
    
    return final_recommendations[:top_n]

In [23]:
def hybrid_recommend(customer_id, top_n=10):

    cf_recs = recommend_products(
        customer_id,
        top_n=20
    )

    cf_scores = {
        sku: score
        for sku, score in cf_recs
    }

    max_cf = max(cf_scores.values())

    cf_scores = {
        sku: score / max_cf
        for sku, score in cf_scores.items()
    }

    customer_profile = retailer_features[
        retailer_features["customerId"]
        == customer_id
    ]

    hub = customer_profile[
        "hub_name"
    ].iloc[0]

    shop_type = customer_profile[
        "shop_type"
    ].iloc[0]

    region_recs = region_trends[
        (
            region_trends["hubName"]
            == hub
        )
        &
        (
            region_trends["shopType"]
            == shop_type
        )
    ]

    region_scores = {}

    max_rank = region_recs[
        "region_rank"
    ].max()

    for _, row in region_recs.iterrows():

        region_scores[
            row["skuNumber"]
        ] = (
            max_rank -
            row["region_rank"]
        ) / max_rank

    pop_scores = popularity_dict

    all_skus = set()

    all_skus.update(cf_scores.keys())
    all_skus.update(region_scores.keys())
    all_skus.update(pop_scores.keys())

    hybrid_scores = {}

    customer_products = set(
        interaction.loc[
            customer_id
        ][
            interaction.loc[
                customer_id
            ] > 0
        ].index
    )

    for sku in all_skus:

        if sku in customer_products:
            continue

        cf = cf_scores.get(sku, 0)
        region = region_scores.get(sku, 0)
        popularity = pop_scores.get(sku, 0)

        final_score = (
            0.6 * cf
            + 0.25 * region
            + 0.15 * popularity
        )

        hybrid_scores[sku] = final_score

    return sorted(
        hybrid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

In [24]:
hybrid_recommend(
    interaction.index[0]
)

[('SKU00075', np.float64(0.8065205938697317)),
 ('SKUPM5', np.float64(0.631061700838619)),
 ('SKU613', np.float64(0.5697265890040937)),
 ('SKU00009', np.float64(0.4423382335142547)),
 ('SKU612', np.float64(0.426491073152642)),
 ('SKU00304', np.float64(0.4131470255796896)),
 ('SKU669', np.float64(0.41113569343674056)),
 ('SKU00033', np.float64(0.3175836492574713)),
 ('SKU00001', np.float64(0.2668446810144503)),
 ('SKU500', np.float64(0.2654428446597701))]

In [25]:
print(interaction.shape)
print(retailer_features.shape)
print(region_trends.shape)
print(similarity_df.shape)
print(rules.shape)

(8640, 252)
(8640, 13)
(10526, 11)
(8640, 8640)
(92, 16)


In [26]:
interaction.shape

(8640, 252)